In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ============================================================
# Complete Data Export for Power BI Dashboard
# ============================================================
import pandas as pd
import numpy as np
import lightgbm as lgb
import catboost as cb
import warnings
warnings.filterwarnings('ignore')

path="/kaggle/input/datasets/tshiamomatshaba/nedbank-transaction-volume-forecasting-challenge/"
print("Loading data...")
train = pd.read_csv(path+'Train (1).csv')
test = pd.read_csv(path+'Test.csv')
txn = pd.read_parquet(path+'transactions_features.parquet')
fin = pd.read_parquet(path+'financials_features.parquet')
demo = pd.read_parquet(path+'demographics_clean.parquet')

CUTOFF = pd.Timestamp('2015-11-01')
txn['TransactionDate'] = pd.to_datetime(txn['TransactionDate'])
txn_hist = txn[txn['TransactionDate'] < CUTOFF].copy()
txn_hist['days_before'] = (CUTOFF - txn_hist['TransactionDate']).dt.days

# ------------------------- 2. Feature Engineering -------------------------
print("Engineering core features...")

def rolling_counts(df, days):
    res = pd.DataFrame()
    res['UniqueID'] = df['UniqueID'].unique()
    for d in days:
        cnt = df[df['days_before'] <= d].groupby('UniqueID').size().rename(f'txn_last_{d}d')
        res = res.merge(cnt, on='UniqueID', how='left')
    return res.fillna(0)

roll = rolling_counts(txn_hist, [30,60,90,180,365])

# Transaction types last 90 days
txn90 = txn_hist[txn_hist['days_before'] <= 90]
types = txn90.groupby(['UniqueID','TransactionTypeDescription']).size().unstack(fill_value=0)
types.columns = [f'type_{c}' for c in types.columns]
types = types.reset_index()

# Seasonal previous year (Nov14-Jan15)
season = txn_hist[(txn_hist['TransactionDate']>=pd.Timestamp('2014-11-01')) & 
                  (txn_hist['TransactionDate']<=pd.Timestamp('2015-01-31'))]
season_cnt = season.groupby('UniqueID').size().rename('prev_year').reset_index()

# Monthly features (optional but good for dashboard exploration)
txn_hist['year_month'] = txn_hist['TransactionDate'].dt.to_period('M')
monthly = txn_hist.groupby(['UniqueID', 'year_month']).size().reset_index(name='monthly_count')
monthly_stats = monthly.groupby('UniqueID')['monthly_count'].agg(
    monthly_mean='mean',
    monthly_std='std',
    monthly_min='min',
    monthly_max='max',
    monthly_last3_mean=lambda x: x.iloc[-3:].mean() if len(x) >= 3 else x.mean(),
    monthly_trend=lambda x: (x.iloc[-3:].mean() - x.iloc[-6:-3].mean()) if len(x) >= 6 else 0,
    monthly_cv=lambda x: x.std() / (x.mean() + 1e-6)
).reset_index()
monthly_stats['monthly_cv'] = monthly_stats['monthly_cv'].replace([np.inf, -np.inf], 0)

# Financial flags
fin_has = fin[['UniqueID']].drop_duplicates()
fin_has['has_financial'] = 1
fin_months = fin.groupby('UniqueID')['RunDate'].nunique().reset_index(name='fin_months_count')

# Merge all features for train and test
def build_feature_matrix(base_df, is_train=True):
    df = base_df.merge(roll, on='UniqueID', how='left')
    df = df.merge(types, on='UniqueID', how='left').fillna(0)
    df = df.merge(season_cnt, on='UniqueID', how='left').fillna(0)
    df = df.merge(monthly_stats, on='UniqueID', how='left').fillna(0)
    df = df.merge(fin_has, on='UniqueID', how='left').fillna(0)
    df = df.merge(fin_months, on='UniqueID', how='left').fillna(0)
    if is_train:
        # For training, we will merge demographics later separately
        pass
    return df

X_train = build_feature_matrix(train[['UniqueID']])
X_test = build_feature_matrix(test[['UniqueID']])

# Income dummies
inc_dummies = pd.get_dummies(demo['IncomeCategory'], prefix='inc')
inc_dummies['UniqueID'] = demo['UniqueID']
X_train = X_train.merge(inc_dummies, on='UniqueID', how='left').fillna(0)
X_test = X_test.merge(inc_dummies, on='UniqueID', how='left').fillna(0)

# Add original demographic columns for dashboard (after merging dummies)
demo_for_dashboard = demo[['UniqueID', 'IncomeCategory', 'OccupationCategory', 'MaritalStatus', 'ResidentialCityName']]

# Transaction type ratios
total = 'txn_last_90d'
type_cols = [c for c in X_train.columns if c.startswith('type_')]
for c in type_cols:
    X_train[f'{c}_ratio'] = X_train[c] / (X_train[total] + 1e-6)
    X_test[f'{c}_ratio'] = X_test[c] / (X_test[total] + 1e-6)
if 'type_Debit' in X_train.columns:
    X_train['debit_ratio'] = X_train['type_Debit'] / (X_train[total] + 1e-6)
    X_test['debit_ratio'] = X_test['type_Debit'] / (X_test[total] + 1e-6)

# Recency
last_txn = txn_hist.groupby('UniqueID')['TransactionDate'].max().reset_index()
last_txn['recency'] = (CUTOFF - last_txn['TransactionDate']).dt.days
X_train = X_train.merge(last_txn[['UniqueID','recency']], on='UniqueID', how='left').fillna(365)
X_test = X_test.merge(last_txn[['UniqueID','recency']], on='UniqueID', how='left').fillna(365)

# Feature columns (excluding UniqueID)
feature_cols = [c for c in X_train.columns if c != 'UniqueID']
X_train_feat = X_train[feature_cols]
X_test_feat = X_test[feature_cols]
y = np.log1p(train['next_3m_txn_count'])

# ------------------------- 3. Train Models -------------------------
print("Training LightGBM...")
lgb_model = lgb.LGBMRegressor(
    n_estimators=180, learning_rate=0.03, num_leaves=31,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    random_state=42, verbose=-1
)
lgb_model.fit(X_train_feat, y)
pred_lgb_test = lgb_model.predict(X_test_feat)
pred_lgb_train = lgb_model.predict(X_train_feat)

print("Training CatBoost...")
cb_model = cb.CatBoostRegressor(
    iterations=500, learning_rate=0.03, depth=6,
    random_seed=42, verbose=False
)
cb_model.fit(X_train_feat, y)
pred_cb_test = cb_model.predict(X_test_feat)
pred_cb_train = cb_model.predict(X_train_feat)

# Blend 60/40
blend_test = 0.6 * pred_lgb_test + 0.4 * pred_cb_test
blend_train = 0.6 * pred_lgb_train + 0.4 * pred_cb_train

# ------------------------- 4. Prepare Dashboard Dataset -------------------------
print("Preparing dashboard CSV...")
# Start with training data
dashboard = train[['UniqueID', 'next_3m_txn_count']].copy()
dashboard['predicted_log'] = blend_train
dashboard['predicted_raw'] = np.expm1(blend_train)  # back to original scale

# Add all core features (for slice/dice in Power BI)
dashboard = dashboard.merge(X_train, on='UniqueID', how='left')

# Add demographics (original categories, not dummies)
dashboard = dashboard.merge(demo_for_dashboard, on='UniqueID', how='left')

# Add monthly stats (already in X_train, but they are included; we can keep them)

# Mark dataset type (train vs test) – we will not include test in dashboard
dashboard['dataset'] = 'train'

# Optionally, add test data without actual target (for completeness)
test_dashboard = test[['UniqueID']].copy()
test_dashboard['next_3m_txn_count'] = np.nan
test_dashboard['predicted_log'] = blend_test
test_dashboard['predicted_raw'] = np.expm1(blend_test)
test_dashboard = test_dashboard.merge(X_test, on='UniqueID', how='left')
test_dashboard = test_dashboard.merge(demo_for_dashboard, on='UniqueID', how='left')
test_dashboard['dataset'] = 'test'

# Combine
full_dashboard = pd.concat([dashboard, test_dashboard], ignore_index=True)

# Save
full_dashboard.to_csv('dashboard_data.csv', index=False)
print("Saved dashboard_data.csv (rows: {})".format(len(full_dashboard)))

# ------------------------- 5. Submission -------------------------
sub = pd.DataFrame({
    'UniqueID': test['UniqueID'],
    'next_3m_txn_count': blend_test
})
sub.to_csv('submission_final.csv', index=False)
print("Saved submission_final.csv")

# ------------------------- 6. Optional: Feature Importance -------------------------
# Feature importance for LightGBM – can be used in Power BI as reference
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)
importance.to_csv('feature_importance.csv', index=False)
print("Saved feature_importance.csv")

print("All done!")

Loading data...
Engineering core features...
Training LightGBM...
Training CatBoost...
Preparing dashboard CSV...
Saved dashboard_data.csv (rows: 11944)
Saved submission_final.csv
Saved feature_importance.csv
All done!
